# Demo — Mapping a Federated Platform to a Governance Operating Model

**Trailhead Provisions** grew by acquisition into four domain teams sharing a data platform.
Before writing a single rule or access policy, a data architect must establish the **operating
model**: which decisions stay central, which belong to domain teams, and who is **Accountable**
for each governance function.

This walkthrough inspects the platform's domains and cross-domain dependencies in your live AWS
Glue Data Catalog, models one row of a RACI matrix, and sketches a slice of the governance
topology; the exercise asks you to justify the full governance model, complete the RACI across all
functions, and draw the complete topology.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

created Glue database 'trailhead'
registered s3://123456789012-trailhead-data with Lake Formation
  customer               rows=  310  table-gaps=[]  untagged-cols=['email']
  orders                 rows= 1200  table-gaps=['owner']  untagged-cols=[]
  order_returns          rows=  106  table-gaps=[]  untagged-cols=[]
  supplier               rows=   40  table-gaps=['retention']  untagged-cols=[]
  product                rows=  200  table-gaps=['retention']  untagged-cols=['unit_cost']
  shipment               rows= 1109  table-gaps=['classification']  untagged-cols=[]
  campaign               rows=   20  table-gaps=['owner']  untagged-cols=[]
  campaign_membership    rows=  800  table-gaps=['owner', 'retention']  untagged-cols=['cust_ref']

Done. Glue database 'trailhead' seeded in s3://123456789012-trailhead-data with Lake Formation enforcement enabled.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

inventory = pd.DataFrame([
    {
        "domain": gc.catalog.table(t).domain,
        "table": t,
        "owner": gc.catalog.table(t).owner or "(no owner set)",
        "classification": gc.catalog.table(t).classification or "(unset)",
    }
    for t in gc.tables()
]).sort_values(["domain", "table"]).reset_index(drop=True)
inventory

Backend: live -> AwsGovernedCatalog (Glue DB: trailhead, Region: us-east-1)


,domain,table,owner,classification
0,Customer Identity & Loyalty,customer,customer-identity@trailhead.example,PII
1,E-commerce Orders,order_returns,orders-team@trailhead.example,PUBLIC
2,E-commerce Orders,orders,(no owner set),SENSITIVE
3,Inventory & Fulfillment,product,inventory-fulfillment@trailhead.example,SENSITIVE
4,Inventory & Fulfillment,shipment,inventory-fulfillment@trailhead.example,(unset)
5,Inventory & Fulfillment,supplier,inventory-fulfillment@trailhead.example,PUBLIC
6,Marketing & Campaigns,campaign,(no owner set),PUBLIC
7,Marketing & Campaigns,campaign_membership,(no owner set),SENSITIVE


## Why pure decentralization fails under GDPR & CSRD
Each domain team runs its own pipelines, yet regulations cross domain boundaries. Inspect the
platform's cross-domain semantic conflicts and pipeline jobs to see where domains collide:

In [3]:
print("--- Cross-domain semantic conflicts ---")
display(gc.semantic_conflicts()[["concept", "domain_a", "domain_b", "impact"]])

print("\n--- Cross-domain pipeline jobs ---")
gc.lineage.events()

--- Cross-domain semantic conflicts ---


,concept,domain_a,domain_b,impact
0,customer identity key,Customer Identity & Loyalty,Marketing & Campaigns,No shared join key; cross-domain customer reso...
1,marketing consent,Customer Identity & Loyalty,Marketing & Campaigns,Two systems of record for consent disagree ~20...
2,carbon unit of measure,Inventory & Fulfillment,Reporting (derived),Mixed units inflate the CSRD disclosure unless...



--- Cross-domain pipeline jobs ---


,job,inputs,outputs
0,build_order_carbon,"orders, shipment",order_carbon
1,build_quarterly_sustainability,order_carbon,quarterly_sustainability
2,build_customer_360,"customer, campaign_membership",customer_360
3,build_marketing_audience,"customer_360, orders",marketing_audience


## How to structure a RACI row (the single-Accountable rule)
In a **federated / hybrid** model, domains are **Responsible (`R`)** for executing governance
on their own tables, but every governance function must have **exactly one Accountable (`A`)**
individual owner so accountability never falls between chairs.

Here is how one governance function row looks before you build the full matrix in the exercise:

| Governance function | E-comm Orders | Customer Identity | Inventory & Fulfillment | Marketing & Campaigns | Accountable (single) |
|---|---|---|---|---|---|
| **Classification taxonomy (`PII`/`SENSITIVE`/`PUBLIC`)** | `R` (tag order tables) | `R` (tag customer PII) | `R` (tag cost/carbon) | `R` (tag campaign tables) | **Central Governance Lead** |

## How to sketch the topology in Mermaid
Use a Mermaid `graph TD` block to show how central standards flow down to domains while regulated
data flows across domains. Here is a two-domain slice illustrating the pattern:

```mermaid
graph TD
    Core["Central Governance Core<br/>(Taxonomy • Consent Standard • Contract Schema)"]
    Cust["Customer Identity & Loyalty<br/>(customer)"]
    Mkt["Marketing & Campaigns<br/>(campaign, campaign_membership)"]

    Core -->|"guardrails & PII standards"| Cust & Mkt
    Cust -->|"loyalty consent"| Core
    Core -->|"authoritative consent SoR"| Mkt
```

**Takeaway / your turn:** in the exercise you'll state and justify Trailhead's governance model,
build the complete RACI across all governance functions (with one `A` per row), and draw the
full four-domain topology including both the GDPR consent flow and the CSRD carbon reporting flow.